In [1]:
import json
import os

from typing import Annotated

from dotenv import load_dotenv

from openai import AsyncOpenAI

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.functions import kernel_function

In [2]:
# 为示例定义一个示例插件
class DestinationsPlugin:
    """假期目的地列表。"""

    @kernel_function(description="提供假期目的地列表。")
    def get_destinations(self) -> Annotated[str, "返回菜单中的特价信息。"]:
        return """
        西班牙巴塞罗那
        法国巴黎
        德国柏林
        日本东京
        美国纽约
        """

    @kernel_function(description="提供目的地的可用航班时间。")
    def get_flight_times(
        self, destination: Annotated[str, "要查询航班时间的目的地。"]
    ) -> Annotated[str, "返回指定目的地的航班时间。"]:
        # 返回 HTTP 错误 404
        return "HTTP ERROR 404: 航班时间服务当前不可用。"

    @kernel_function(description="提供目的地的可用航班时间的备份函数。")
    def get_flight_times_backup(
        self, destination: Annotated[str, "要查询航班时间的目的地。"]
    ) -> Annotated[str, "返回指定目的地的航班时间。"]:
        flight_times = {
            "Barcelona": ["08:30 AM", "02:15 PM", "10:45 PM"],
            "Paris": ["06:45 AM", "12:30 PM", "07:15 PM"],
            "Berlin": ["07:20 AM", "01:45 PM", "09:30 PM"],
            "Tokyo": ["11:00 AM", "05:30 PM", "11:55 PM"],
            "New York": ["05:15 AM", "03:00 PM", "08:45 PM"]
        }

        # 从可能包含国家信息的输入中提取城市名称
        city = destination.split(',')[0].strip()

        if city in flight_times:
            times = ", ".join(flight_times[city])
            return f"{city} 的航班时间：{times}"
        else:
            return f"没有 {city} 的航班信息。"

In [8]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("API_URL"),
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id=os.getenv("MODEL_FREE_8B"),
    # async_client=client,
    api_key=os.getenv("API_KEY"),
)

In [5]:
AGENT_NAME = "TravelAgent"
AGENT_INSTRUCTIONS = """ \
"你是航班预订代理，提供可用航班信息，并在被询问时给出旅行活动建议。
旅行活动建议应该针对客户、地点和在地点停留的时间。

你可以访问以下工具来帮助用户规划他们的旅行：
1. get_destinations：返回用户可以选择的可用假期目的地列表。
2. get_flight_times：为特定目的地提供可用航班时间。
3. get_flight_times_backup：当主服务不可用时提供可用航班时间的备份函数。

你协助用户的流程：
- 当用户询问航班预订时，使用 get_flight_times 为他们选择的目的地预订最早的可用航班。
- 如果 get_flight_times 返回错误消息，立即使用相同的目的地参数调用 get_flight_times_backup 来检索航班信息。
- 由于你无法访问预订系统，不要要求继续预订，只需假设你已经预订了航班。
- 使用过去的对话历史来了解用户偏好，并在提出航班和活动建议时考虑这些偏好。当提出建议时，如果基于用户偏好，请非常清楚地说明你提出这个建议的原因。

指导原则：
- 使用工具时使用确切的目的地名称（巴塞罗那、巴黎、柏林、东京、纽约）
- 以乐于助人和热情的方式回应旅行可能性
- 始终寻求反馈，确保你的建议符合用户的期望
- 当请求超出你的能力范围时，请承认
- 为了更好的格式，始终以列表格式显示航班时间
- 给出任何时间建议时，考虑时间框架是否合理。如果不合理，请再次回应。
- 如果航班时间服务不可用，请告知用户你正在使用备份航班数据，同时保持积极的语气。

你的目标是通过了解用户偏好并提供量身定制的建议，帮助用户高效地探索假期选项并做出明智的旅行决策。
"""
# 创建代理
agent = ChatCompletionAgent(
    service=chat_completion_service,
    plugins=[DestinationsPlugin()],
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
)

In [9]:
from IPython.display import display, HTML

user_inputs = [
    "Book me a flight to Barcelona",
]

# 创建一个线程来保存对话
# 如果没有提供线程，将创建一个新线程
# 并在初始响应中返回
thread: ChatHistoryAgentThread | None = None

async def main():
    global thread
    
    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>用户：</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # 用于重建流式函数调用的缓冲区
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # 累积参数（以流式方式分块传输）
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # 在显示结果之前完成任何待处理的函数调用
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # 保留为原始字符串

                        function_calls.append(f"调用函数：{current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\n函数结果：\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>函数调用（点击展开）</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; "
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or '助手'}：</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()


ServiceResponseException: ("<class 'semantic_kernel.connectors.ai.open_ai.services.open_ai_chat_completion.OpenAIChatCompletion'> service failed to complete the prompt", APITimeoutError('Request timed out.'))